# SSIF_V3：Google Colab 中文實作教學

本 Notebook 使用研究資料：

```python
data_path = '/content/drive/MyDrive/00_SSIF/combined_data.csv'
```

流程為：掛載 Drive → 下載程式 → 檢查 CSV 欄位 → 將 CSV 轉為 SSIF event JSON → 資料稽核與事件切分 → 快速訓練 → 正式訓練 → 外部評估。

> `prepare_ssif_dataset.py` 與 `train_ssif_v3.py` 讀取的是「每個事件一個 JSON」的資料目錄，因此 `combined_data.csv` 不可直接當作 `--data-dir`。本 Notebook 先使用 `csv_to_ssif_json.py` 轉換。

## 1. 掛載 Google Drive、clone 程式並安裝套件

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
%cd /content
!rm -rf SSIF_V3
!git clone https://github.com/oceanicdayi/SSIF_V3.git
%cd /content/SSIF_V3
!git rev-parse HEAD
!python -m pip install -q -r requirements.txt

## 2. 設定研究資料與輸出路徑

In [ ]:
from pathlib import Path
import json
import subprocess
import pandas as pd
import torch
from IPython.display import display

data_path = '/content/drive/MyDrive/00_SSIF/combined_data.csv'
DATA_CSV = Path(data_path)

REPO_ROOT = Path('/content/SSIF_V3')
WORK_ROOT = Path('/content/drive/MyDrive/00_SSIF/SSIF_V3_workspace')

# combined_data.csv 轉換後的 event JSON 訓練資料
TRAIN_DATA = WORK_ROOT / 'data' / 'training_archive_json'

# 獨立外部評估資料必須另外準備；不可自動從同一 CSV 當作 external test
EXTERNAL_DATA = WORK_ROOT / 'data' / 'external_evaluation_json'

PREPARED_DIR = WORK_ROOT / 'prepared' / 'split_v1'
MODEL_DIR = WORK_ROOT / 'models' / 'seed_20260728'
INFERENCE_DIR = WORK_ROOT / 'inference' / 'external_seed_20260728'
REPLAY_DIR = WORK_ROOT / 'replay'

WINDOWS = [10, 15, 20, 25, 30, 35, 40]
SEED = 20260728

for path in [
    TRAIN_DATA, EXTERNAL_DATA, PREPARED_DIR,
    MODEL_DIR, INFERENCE_DIR, REPLAY_DIR
]:
    path.mkdir(parents=True, exist_ok=True)

assert DATA_CSV.is_file(), f'找不到研究資料：{DATA_CSV}'
print('CSV path:', DATA_CSV)
print('CSV size (GB):', round(DATA_CSV.stat().st_size / 1024**3, 3))
print('GPU available:', torch.cuda.is_available())
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU')

## 3. 只讀取前幾列，檢查大型 CSV 的欄位

In [ ]:
preview = pd.read_csv(DATA_CSV, nrows=5, low_memory=False)
print('Columns:')
print(preview.columns.tolist())
display(preview)

## 4. 自動偵測 CSV 格式與欄位

轉換器支援：

- `wide`：每個 event–station 一列，震度欄位為 `t1...t120`、`sec_1...sec_120` 等。
- `sequence`：每個 event–station 一列，其中一欄是長度 120 的 list/JSON sequence。
- `long`：每列為 event–station–second–intensity。

In [ ]:
inspect_cmd = [
    'python', 'csv_to_ssif_json.py', 'inspect',
    '--csv', str(DATA_CSV),
    '--rows', '5',
    '--horizon', '120',
]
result = subprocess.run(
    inspect_cmd, cwd=REPO_ROOT, check=True,
    capture_output=True, text=True
)
schema = json.loads(result.stdout)
print(json.dumps(schema['detected'], ensure_ascii=False, indent=2))

## 5. 確認欄位映射後轉成 event JSON

先查看上一格的 `detected`。自動偵測正確時，只需把 `RUN_CSV_CONVERSION=True`。

自動偵測不正確時，填入實際欄位名稱，例如：

```python
EVENT_COL = 'event_id'
STATION_COL = 'station_id'
SEQUENCE_COL = 'intensity_series'
```

長表則設定 `SECOND_COL` 與 `VALUE_COL`。728 MB CSV 轉換需要一段時間，輸出會保存在 Google Drive。

In [ ]:
RUN_CSV_CONVERSION = False

CSV_LAYOUT = 'auto'       # auto / wide / sequence / long
EVENT_COL = None
STATION_COL = None
SEQUENCE_COL = None
SECOND_COL = None
VALUE_COL = None
ORIGIN_COL = None
MAGNITUDE_COL = None
DEPTH_COL = None
LONGITUDE_COL = None
LATITUDE_COL = None
DISTANCE_COL = None

if RUN_CSV_CONVERSION:
    convert_cmd = [
        'python', 'csv_to_ssif_json.py', 'convert',
        '--csv', str(DATA_CSV),
        '--output-dir', str(TRAIN_DATA),
        '--layout', CSV_LAYOUT,
        '--horizon', '120',
        '--chunk-size', '5000',
        '--overwrite',
    ]

    optional_args = {
        '--event-col': EVENT_COL,
        '--station-col': STATION_COL,
        '--sequence-col': SEQUENCE_COL,
        '--second-col': SECOND_COL,
        '--value-col': VALUE_COL,
        '--origin-col': ORIGIN_COL,
        '--magnitude-col': MAGNITUDE_COL,
        '--depth-col': DEPTH_COL,
        '--longitude-col': LONGITUDE_COL,
        '--latitude-col': LATITUDE_COL,
        '--distance-col': DISTANCE_COL,
    }
    for flag, value in optional_args.items():
        if value is not None:
            convert_cmd.extend([flag, value])

    print('Running:', ' '.join(convert_cmd))
    subprocess.run(convert_cmd, cwd=REPO_ROOT, check=True)
else:
    print('請先核對欄位偵測結果，再將 RUN_CSV_CONVERSION 改為 True。')

## 6. 確認轉換結果

In [ ]:
train_files = sorted(TRAIN_DATA.rglob('event_*.json'))
external_files = sorted(EXTERNAL_DATA.rglob('*.json'))

print('Converted training JSON:', len(train_files))
print('External evaluation JSON:', len(external_files))
print('Training examples:', train_files[:3])

summary_path = TRAIN_DATA / 'conversion_summary.json'
if summary_path.exists():
    conversion_summary = json.loads(summary_path.read_text(encoding='utf-8'))
    print(json.dumps(conversion_summary, ensure_ascii=False, indent=2))

## 7. 執行合成資料 smoke test

In [ ]:
%cd /content/SSIF_V3
!python smoke_test_pipeline_v3.py

## 8. 稽核資料並建立 train／validation／calibration／test

In [ ]:
RUN_AUDIT_SPLIT = False

if RUN_AUDIT_SPLIT:
    audit_cmd = [
        'python', 'prepare_ssif_dataset.py', 'audit-split',
        '--data-dir', str(TRAIN_DATA),
        '--output-dir', str(PREPARED_DIR),
        '--windows', *map(str, WINDOWS),
        '--label-horizon', '120',
        '--min-label-valid-fraction', '0.80',
        '--min-window-valid-fraction', '0.80',
        '--train-ratio', '0.70',
        '--validation-ratio', '0.10',
        '--calibration-ratio', '0.10',
        '--test-ratio', '0.10',
        '--split-candidates', '5000',
        '--seed', str(SEED),
    ]
    subprocess.run(audit_cmd, cwd=REPO_ROOT, check=True)
else:
    print('完成 CSV 轉換並人工檢查後，再將 RUN_AUDIT_SPLIT 改為 True。')

## 9. 顯示稽核與事件切分

In [ ]:
if (PREPARED_DIR / 'event_split.csv').exists():
    event_split = pd.read_csv(PREPARED_DIR / 'event_split.csv')
    display(event_split['split'].value_counts().rename_axis('split').to_frame('events'))
    display(event_split.head())

if (PREPARED_DIR / 'audit_summary.json').exists():
    audit_summary = json.loads(
        (PREPARED_DIR / 'audit_summary.json').read_text(encoding='utf-8')
    )
    print(json.dumps(audit_summary, ensure_ascii=False, indent=2))

## 10. EW10 一個 epoch 快速訓練

In [ ]:
RUN_QUICK_TRAIN = False
QUICK_MODEL_DIR = WORK_ROOT / 'models' / 'quick_EW10'

if RUN_QUICK_TRAIN:
    quick_cmd = [
        'python', 'train_ssif_v3.py', 'train-all',
        '--data-dir', str(TRAIN_DATA),
        '--split-manifest', str(PREPARED_DIR / 'split_manifest.json'),
        '--output-dir', str(QUICK_MODEL_DIR),
        '--windows', '10',
        '--label-horizon', '120',
        '--cohort', 'common',
        '--epochs', '1',
        '--batch-size', '16',
        '--eval-batch-size', '64',
        '--lr', '3e-4',
        '--seed', str(SEED),
        '--window-seed-mode', 'same',
    ]
    if torch.cuda.is_available():
        quick_cmd.append('--amp')
    subprocess.run(quick_cmd, cwd=REPO_ROOT, check=True)

## 11. 正式訓練 EW10–EW40

In [ ]:
RUN_FULL_TRAIN = False

if RUN_FULL_TRAIN:
    train_cmd = [
        'python', 'train_ssif_v3.py', 'train-all',
        '--data-dir', str(TRAIN_DATA),
        '--split-manifest', str(PREPARED_DIR / 'split_manifest.json'),
        '--output-dir', str(MODEL_DIR),
        '--windows', *map(str, WINDOWS),
        '--label-horizon', '120',
        '--cohort', 'common',
        '--epochs', '30',
        '--batch-size', '16',
        '--eval-batch-size', '64',
        '--lr', '3e-4',
        '--weight-decay', '1e-2',
        '--warmup-ratio', '0.10',
        '--min-precision', '0.90',
        '--seed', str(SEED),
        '--window-seed-mode', 'same',
        '--patience', '6',
        '--workers', '2',
    ]
    if torch.cuda.is_available():
        train_cmd.append('--amp')
    subprocess.run(train_cmd, cwd=REPO_ROOT, check=True)

## 12. 獨立外部評估

`EXTERNAL_DATA` 必須是未參與模型開發的獨立事件資料。若目前只有 `combined_data.csv`，先完成四集合 internal test；不要把同一份資料複製到 external evaluation 再宣稱外部驗證。

In [ ]:
RUN_EXTERNAL_EVAL = False

if RUN_EXTERNAL_EVAL:
    assert external_files, 'EXTERNAL_DATA 尚未放入獨立事件 JSON'
    eval_cmd = [
        'python', 'train_ssif_v3.py', 'evaluate-all',
        '--data-dir', str(EXTERNAL_DATA),
        '--model-root', str(MODEL_DIR),
        '--output-dir', str(INFERENCE_DIR),
        '--windows', *map(str, WINDOWS),
        '--label-horizon', '120',
        '--cohort', 'common',
        '--batch-size', '128',
        '--workers', '2',
    ]
    subprocess.run(eval_cmd, cwd=REPO_ROOT, check=True)

## 13. 查看模型結果

In [ ]:
if (MODEL_DIR / 'summary.json').exists():
    training_summary = json.loads(
        (MODEL_DIR / 'summary.json').read_text(encoding='utf-8')
    )
    rows = []
    for item in training_summary:
        alert = item['test']['alert']
        rows.append({
            'window': item['window'],
            'best_epoch': item['best_epoch'],
            'threshold': item['threshold'],
            'precision': alert['precision'],
            'pod': alert['pod'],
            'f1': alert['f1'],
            'fpr': alert['fpr'],
        })
    display(pd.DataFrame(rows).sort_values('window'))

## 研究注意事項

1. `combined_data.csv` 是原始研究資料；正式模型輸入仍是轉換後的事件 JSON。
2. `conversion_summary.json`、`event_index.csv`、`split_manifest.json` 與 Git commit SHA 都要保存。
3. 只修改資料內容或欄位映射，就應重新產生資料 fingerprint 與 split 版本。
4. validation 選 epoch；calibration 決定警報 threshold；test 僅在兩者固定後使用。
5. 獨立 external evaluation 不可與 `combined_data.csv` 重複。